> **Nota metodológica (UCM):** este notebook es **diagnóstico**.  
> Si el cruce por código (`CodigoExterno` ↔ `CodigoLicitacion`) es bajo, eso **no es error**: significa que la trazabilidad contractual directa es **parcial** y no generalizable.  
> El CRUCE oficial para BI se mantiene como **reconciliación semántica-temporal (lag)**.

# 06 — Diagnóstico de coincidencias LIC ↔ OC (3 meses) — Fuente CSV

**TFM ChileCompra Data Platform (UCM) — Auditoría técnica**  
**Objetivo:** evaluar **si existe** (y con qué cobertura) una coincidencia **directa** entre LIC y OC en los CSV fuente
para los meses **2024-10, 2024-11, 2024-12**, sin suposiciones y **sin modificar datos**.

> ⚠️ Este notebook **NO** intenta construir una relación contractual universal.  
> Su rol es **cerrar la “espina”**: descartar (o cuantificar) coincidencias directas por campos como `CodigoLicitacion`
y por referencias embebidas en textos.

## Qué produce
- Tabla por mes con **match exacto** (LIC↔OC) usando columnas candidatas.
- Diagnóstico de **tipos/normalización** (texto vs número, espacios, mayúsculas, guiones).
- Diagnóstico de coincidencia por **proveedor** y **organismo** como apoyo (no como FK).
- Diagnóstico de coincidencias por **extracción desde textos** en OC (si aparece el código LIC dentro de `Nombre/Descripción`).

## Ubicación esperada de datos (según tu repo)
Base: `/home/engineer/Documents/Proyectos/TFM_CSV_Biblioteca_Readme`
- LIC: `01_LIC/LIC_YYYY-MM.csv`
- OC:  `02_OC/OC_YYYY-MM.csv`


In [1]:
# 1 - SETUP (parámetros y librerías)
import os
import re
import csv
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)

BASE_DIR = "/home/engineer/Documents/Proyectos/TFM_CSV_Biblioteca_Readme"
DIR_LIC = os.path.join(BASE_DIR, "01_LIC")
DIR_OC  = os.path.join(BASE_DIR, "02_OC")

MESES = ["2024-10", "2024-11", "2024-12"]

# Muestras (para no reventar RAM)
NROWS_KEYS = 300_000         # lectura para llaves (ajusta si necesitas)
NROWS_TEXT = 150_000         # lectura para texto (extracción de códigos)

ENCODINGS_TRY = ("utf-8", "latin1")

print("BASE_DIR:", BASE_DIR)
print("LIC:", DIR_LIC)
print("OC :", DIR_OC)


BASE_DIR: /home/engineer/Documents/Proyectos/TFM_CSV_Biblioteca_Readme
LIC: /home/engineer/Documents/Proyectos/TFM_CSV_Biblioteca_Readme/01_LIC
OC : /home/engineer/Documents/Proyectos/TFM_CSV_Biblioteca_Readme/02_OC


In [2]:
# 2 - UTILIDADES robustas (detectar separador/encoding, leer header, normalizar strings)
def detect_sep_and_encoding(path, encodings=ENCODINGS_TRY):
    last_err = None
    for enc in encodings:
        try:
            with open(path, "r", encoding=enc, errors="strict", newline="") as f:
                first_line = f.readline().strip("\n\r")
            semi = first_line.count(";")
            comma = first_line.count(",")
            sep = ";" if semi > comma else ","
            cols = next(csv.reader([first_line], delimiter=sep))
            cols = [c.strip().strip('"').strip("'") for c in cols]
            return enc, sep, cols
        except Exception as e:
            last_err = e
            continue
    raise RuntimeError(f"No pude leer header {os.path.basename(path)}. Último error: {last_err}")

def read_csv_safe(path, usecols=None, nrows=None):
    enc, sep, cols = detect_sep_and_encoding(path)
    if usecols is not None:
        usecols = [c.strip().strip('"').strip("'") for c in usecols]
    df = pd.read_csv(
        path,
        usecols=usecols,
        nrows=nrows,
        encoding=enc,
        sep=sep,
        engine="python",
        on_bad_lines="skip"
    )
    df.columns = [c.strip().strip('"').strip("'") for c in df.columns]
    return df, {"encoding": enc, "sep": sep, "ncols": len(cols)}

def norm_key(x):
    if x is None:
        return None
    s = str(x).strip()
    if s == "" or s.lower() in {"nan", "none", "null"}:
        return None
    s = s.strip('"').strip("'").strip()
    s = re.sub(r"\s+", "", s)
    s = s.upper()
    return s


def norm_digits(x):
    """Normalización para códigos numéricos (proveedor/organismo):
    - '287333.0' -> '287333'
    - elimina espacios y separadores
    - conserva solo dígitos
    """
    if x is None:
        return None
    s = str(x).strip()
    if s == "" or s.lower() in {"nan", "none", "null"}:
        return None
    s = s.strip('"').strip("'").strip()
    if s.endswith(".0"):
        s = s[:-2]
    digits = re.findall(r"\d+", s)
    if not digits:
        return None
    return "".join(digits)

def find_cols_by_patterns(cols, patterns):
    cols_l = {c.lower(): c for c in cols}
    out = []
    for cl, orig in cols_l.items():
        if any(p in cl for p in patterns):
            out.append(orig)
    return out

def pick_first(cols, preferred_tokens):
    cols_l = {c.lower(): c for c in cols}
    for token in preferred_tokens:
        for cl, orig in cols_l.items():
            if token in cl:
                return orig
    return None

print("OK - utilidades cargadas")


OK - utilidades cargadas


In [3]:
# 3 - Descubrir columnas candidatas por mes (LIC y OC)
PATRONES_LIC = ["codigoextern", "licit", "codigo", "id", "link"]
PATRONES_OC  = ["codigolicit", "licit", "codigo", "extern", "descripcion", "nombre", "observ", "especificacion"]

def scan_month(mes):
    lic_file = os.path.join(DIR_LIC, f"LIC_{mes}.csv")
    oc_file  = os.path.join(DIR_OC,  f"OC_{mes}.csv")
    if not (os.path.exists(lic_file) and os.path.exists(oc_file)):
        return {"mes": mes, "status": "MISSING_FILE", "lic": lic_file, "oc": oc_file}

    enc_l, sep_l, cols_l = detect_sep_and_encoding(lic_file)
    enc_o, sep_o, cols_o = detect_sep_and_encoding(oc_file)

    cand_lic = find_cols_by_patterns(cols_l, PATRONES_LIC)
    cand_oc  = find_cols_by_patterns(cols_o, PATRONES_OC)

    col_lic_main = pick_first(cols_l, ["codigoexterno"]) or pick_first(cols_l, ["codigoextern"])
    col_oc_main  = pick_first(cols_o, ["codigolicit"]) or pick_first(cols_o, ["codigo licit"])

    return {
        "mes": mes, "status": "OK",
        "lic_file": os.path.basename(lic_file),
        "oc_file": os.path.basename(oc_file),
        "lic_meta": {"encoding": enc_l, "sep": sep_l, "ncols": len(cols_l)},
        "oc_meta": {"encoding": enc_o, "sep": sep_o, "ncols": len(cols_o)},
        "lic_main": col_lic_main,
        "oc_main": col_oc_main,
        "lic_candidates": cand_lic[:30],
        "oc_candidates": cand_oc[:30],
    }

scan = [scan_month(m) for m in MESES]
df_scan = pd.DataFrame(scan)
display(df_scan[["mes","status","lic_main","oc_main"]])


,mes,status,lic_main,oc_main
0,2024-10,OK,CodigoExterno,CodigoLicitacion
1,2024-11,OK,CodigoExterno,CodigoLicitacion
2,2024-12,OK,CodigoExterno,CodigoLicitacion


In [4]:
# 4 - Matching EXACTO: CodigoExterno (LIC) vs CodigoLicitacion (OC) si existen (por mes)
def match_exact(mes, col_lic, col_oc):
    lic_path = os.path.join(DIR_LIC, f"LIC_{mes}.csv")
    oc_path  = os.path.join(DIR_OC,  f"OC_{mes}.csv")

    lic_df, lic_meta = read_csv_safe(lic_path, usecols=[col_lic], nrows=NROWS_KEYS)
    oc_df,  oc_meta  = read_csv_safe(oc_path,  usecols=[col_oc],  nrows=NROWS_KEYS)

    lic_s = lic_df[col_lic].map(norm_key).dropna()
    oc_s  = oc_df[col_oc].map(norm_key).dropna()

    set_lic = set(lic_s.unique())
    set_oc  = set(oc_s.unique())
    inter = set_lic & set_oc

    return {
        "mes": mes,
        "col_lic": col_lic,
        "col_oc": col_oc,
        "lic_keys": len(set_lic),
        "oc_keys": len(set_oc),
        "matches": len(inter),
        "pct_match_lic": (len(inter)/len(set_lic)*100) if set_lic else 0.0,
        "pct_match_oc":  (len(inter)/len(set_oc)*100) if set_oc else 0.0,
        "ejemplos": list(sorted(inter))[:10],
        "lic_meta": lic_meta,
        "oc_meta": oc_meta,
    }

rows = []
for s in scan:
    if s.get("status") != "OK":
        rows.append({"mes": s["mes"], "status": s["status"]})
        continue
    if not s.get("lic_main") or not s.get("oc_main"):
        rows.append({"mes": s["mes"], "status": "NO_MAIN_COL", "lic_main": s.get("lic_main"), "oc_main": s.get("oc_main")})
        continue
    try:
        r = match_exact(s["mes"], s["lic_main"], s["oc_main"])
        r["status"] = "OK"
        rows.append(r)
    except Exception as e:
        rows.append({"mes": s["mes"], "status": "ERROR", "error": repr(e), "lic_main": s.get("lic_main"), "oc_main": s.get("oc_main")})

df_exact = pd.DataFrame(rows)
display(df_exact[["mes","status","col_lic","col_oc","lic_keys","oc_keys","matches","pct_match_lic","pct_match_oc"]])


,mes,status,col_lic,col_oc,lic_keys,oc_keys,matches,pct_match_lic,pct_match_oc
0,2024-10,OK,CodigoExterno,CodigoLicitacion,15410,22653,363,2.355613,1.602437
1,2024-11,OK,CodigoExterno,CodigoLicitacion,7105,22201,365,5.137227,1.644070
2,2024-12,OK,CodigoExterno,CodigoLicitacion,8215,24272,454,5.526476,1.870468


## 5 — Diagnóstico de “errores por tipos” (texto vs numérico)

Si un cruce falla, típicamente es por:
- espacios / mayúsculas / comillas
- normalización agresiva (p.ej. borrar guiones del código)
- casting a int/float (incorrecto para códigos tipo `1234-1-LP24`)

El siguiente bloque muestra ejemplos crudos vs normalizados.


In [5]:
# 5 - Mostrar ejemplos crudos vs normalizados (si hay columnas main)
def sample_values(mes, col_lic, col_oc, k=12):
    lic_path = os.path.join(DIR_LIC, f"LIC_{mes}.csv")
    oc_path  = os.path.join(DIR_OC,  f"OC_{mes}.csv")

    lic_df, _ = read_csv_safe(lic_path, usecols=[col_lic], nrows=2000)
    oc_df,  _ = read_csv_safe(oc_path,  usecols=[col_oc],  nrows=2000)

    lic_raw = lic_df[col_lic].dropna().astype(str).head(k).tolist()
    oc_raw  = oc_df[col_oc].dropna().astype(str).head(k).tolist()

    lic_norm = [norm_key(x) for x in lic_raw]
    oc_norm  = [norm_key(x) for x in oc_raw]

    return pd.DataFrame({"LIC_raw": lic_raw, "LIC_norm": lic_norm, "OC_raw": oc_raw, "OC_norm": oc_norm})

for s in scan:
    if s.get("status") == "OK" and s.get("lic_main") and s.get("oc_main"):
        print("\n=== Ejemplos", s["mes"], "===")
        display(sample_values(s["mes"], s["lic_main"], s["oc_main"]))



=== Ejemplos 2024-10 ===


,LIC_raw,LIC_norm,OC_raw,OC_norm
0,5184-128-LR23,5184-128-LR23,2684-36-LR22,2684-36-LR22
1,5056-104-LP23,5056-104-LP23,4547-27-LP22,4547-27-LP22
2,5056-104-LP23,5056-104-LP23,2943-3-LR23,2943-3-LR23
3,2111-3-LQ24,2111-3-LQ24,4121-7-LE23,4121-7-LE23
4,2111-3-LQ24,2111-3-LQ24,898-64-LP23,898-64-LP23
5,2111-3-LQ24,2111-3-LQ24,898-64-LP23,898-64-LP23
6,2111-3-LQ24,2111-3-LQ24,1057501-356-LQ22,1057501-356-LQ22
7,2111-3-LQ24,2111-3-LQ24,1057501-356-LQ22,1057501-356-LQ22
8,2111-3-LQ24,2111-3-LQ24,898-103-LP23,898-103-LP23
9,2111-3-LQ24,2111-3-LQ24,655-20-LE22,655-20-LE22



=== Ejemplos 2024-11 ===


,LIC_raw,LIC_norm,OC_raw,OC_norm
0,1003473-4-LR24,1003473-4-LR24,2560-23-LR20,2560-23-LR20
1,1003473-4-LR24,1003473-4-LR24,2309-21-LE22,2309-21-LE22
2,979-3-LQ24,979-3-LQ24,2699-2-LR23,2699-2-LR23
3,2424-23-LE24,2424-23-LE24,2699-2-LR23,2699-2-LR23
4,2424-23-LE24,2424-23-LE24,3799-80-LE22,3799-80-LE22
5,2424-23-LE24,2424-23-LE24,953-25-LQ23,953-25-LQ23
6,2424-23-LE24,2424-23-LE24,953-25-LQ23,953-25-LQ23
7,1966-3-O124,1966-3-O124,953-25-LQ23,953-25-LQ23
8,1966-3-O124,1966-3-O124,953-25-LQ23,953-25-LQ23
9,1966-3-O124,1966-3-O124,953-25-LQ23,953-25-LQ23



=== Ejemplos 2024-12 ===


,LIC_raw,LIC_norm,OC_raw,OC_norm
0,1057439-111-LR23,1057439-111-LR23,2560-20-LR19,2560-20-LR19
1,1057439-111-LR23,1057439-111-LR23,2560-38-LR20,2560-38-LR20
2,1242132-1-LP24,1242132-1-LP24,2560-34-LR20,2560-34-LR20
3,1242132-1-LP24,1242132-1-LP24,3612-26-CO21,3612-26-CO21
4,1242132-1-LP24,1242132-1-LP24,1532-32-LE22,1532-32-LE22
5,1966-1-O124,1966-1-O124,1099834-8-LE23,1099834-8-LE23
6,1966-1-O124,1966-1-O124,2859-48-LR23,2859-48-LR23
7,1966-1-O124,1966-1-O124,3824-53-LE23,3824-53-LE23
8,1966-1-O124,1966-1-O124,1058071-33-LE23,1058071-33-LE23
9,1966-1-O124,1966-1-O124,1641-182-LP21,1641-182-LP21


## 6 — Coincidencia por texto (cuando OC no trae la columna o viene incompleta)

Extrae códigos tipo `NNNN-N-TTYY` desde columnas textuales OC (Nombre/Descripción/Observaciones)
y cruza contra `CodigoExterno` de LIC del mismo mes.

Esto sirve para detectar casos donde el código LIC existe, pero no en una columna estructurada.


In [6]:
# 6 - Extracción de códigos LIC desde textos OC y match contra LIC (mismo mes)
re_code = re.compile(r"\b\d{3,7}-\d{1,4}-[A-Z]{1,4}\d{2}\b")
TEXT_COL_TOKENS = ["nombre", "descripcion", "observ", "especificacion"]

def extract_codes(series):
    out = []
    for v in series.dropna().astype(str):
        out.extend(re_code.findall(v.upper()))
    return out

def text_match_month(mes, col_lic):
    lic_path = os.path.join(DIR_LIC, f"LIC_{mes}.csv")
    oc_path  = os.path.join(DIR_OC,  f"OC_{mes}.csv")

    lic_df, _ = read_csv_safe(lic_path, usecols=[col_lic], nrows=NROWS_TEXT)
    lic_keys = set(lic_df[col_lic].map(norm_key).dropna().unique())

    _, _, oc_cols = detect_sep_and_encoding(oc_path)
    text_cols = [c for c in oc_cols if any(t in c.lower() for t in TEXT_COL_TOKENS)][:6]

    oc_df, _ = read_csv_safe(oc_path, usecols=text_cols, nrows=NROWS_TEXT)

    extracted = []
    for c in text_cols:
        extracted.extend(extract_codes(oc_df[c]))

    set_ext = set([norm_key(x) for x in extracted if x])
    inter = set_ext & lic_keys

    return {
        "mes": mes,
        "text_cols_used": text_cols,
        "extracted_codes_unique": len(set_ext),
        "matches_with_LIC": len(inter),
        "pct_match_over_extracted": (len(inter)/len(set_ext)*100) if set_ext else 0.0,
        "ejemplos_match": list(sorted(inter))[:10],
    }

rows = []
for s in scan:
    if s.get("status") != "OK" or not s.get("lic_main"):
        continue
    try:
        rows.append(text_match_month(s["mes"], s["lic_main"]))
    except Exception as e:
        rows.append({"mes": s["mes"], "status":"ERROR", "error": repr(e)})

df_text = pd.DataFrame(rows)
display(df_text)


,mes,text_cols_used,extracted_codes_unique,matches_with_LIC,pct_match_over_extracted,ejemplos_match
0,2024-10,"[Nombre, Descripcion/Obervaciones, Descripcion...",36804,0,0.0,[]
1,2024-11,"[Nombre, Descripcion/Obervaciones, Descripcion...",36214,0,0.0,[]
2,2024-12,"[Nombre, Descripcion/Obervaciones, Descripcion...",35400,0,0.0,[]


## 7 — Proveedor y Organismo (apoyo, NO FK)

Mide intersecciones de conjuntos por mes para columnas candidatas de proveedor y organismo.
Sirve para entender sesgos (por ejemplo: matches solo en ciertos proveedores).


In [7]:
# 7 - Proveedor/Organismo: detectar columnas candidatas y medir intersección (diagnóstico)
TOK_PROV = ["codigoproveedor", "proveedor"]
TOK_ORG_LIC = ["codigoorganismo", "organismo"]
TOK_ORG_OC  = ["codigoorganismopublico", "codigoorganismo", "organismo"]

def pick_candidate(cols, tokens):
    cols_l = {c.lower().replace(" ", ""): c for c in cols}
    for tok in tokens:
        t = tok.replace(" ", "")
        for cl, orig in cols_l.items():
            if t in cl:
                return orig
    return None

def diag_sets(mes):
    lic_path = os.path.join(DIR_LIC, f"LIC_{mes}.csv")
    oc_path  = os.path.join(DIR_OC,  f"OC_{mes}.csv")

    _, _, lic_cols = detect_sep_and_encoding(lic_path)
    _, _, oc_cols  = detect_sep_and_encoding(oc_path)

    lic_prov = pick_candidate(lic_cols, TOK_PROV)
    oc_prov  = pick_candidate(oc_cols, TOK_PROV)
    lic_org  = pick_candidate(lic_cols, TOK_ORG_LIC)
    oc_org   = pick_candidate(oc_cols, TOK_ORG_OC)

    out = {"mes": mes, "lic_prov": lic_prov, "oc_prov": oc_prov, "lic_org": lic_org, "oc_org": oc_org}

    if lic_prov and oc_prov:
        lic_df, _ = read_csv_safe(lic_path, usecols=[lic_prov], nrows=NROWS_KEYS)
        oc_df,  _ = read_csv_safe(oc_path,  usecols=[oc_prov],  nrows=NROWS_KEYS)
        set_l = set(lic_df[lic_prov].map(norm_digits).dropna().unique())
        set_o = set(oc_df[oc_prov].map(norm_digits).dropna().unique())
        inter = set_l & set_o
        out.update({
            "prov_lic": len(set_l), "prov_oc": len(set_o), "prov_inter": len(inter),
            "prov_pct_lic": (len(inter)/len(set_l)*100) if set_l else 0.0
        })

    if lic_org and oc_org:
        lic_df, _ = read_csv_safe(lic_path, usecols=[lic_org], nrows=NROWS_KEYS)
        oc_df,  _ = read_csv_safe(oc_path,  usecols=[oc_org],  nrows=NROWS_KEYS)
        set_l = set(lic_df[lic_org].map(norm_digits).dropna().unique())
        set_o = set(oc_df[oc_org].map(norm_digits).dropna().unique())
        inter = set_l & set_o
        out.update({
            "org_lic": len(set_l), "org_oc": len(set_o), "org_inter": len(inter),
            "org_pct_lic": (len(inter)/len(set_l)*100) if set_l else 0.0
        })

    return out

rows = []
for m in MESES:
    try:
        rows.append(diag_sets(m))
    except Exception as e:
        rows.append({"mes": m, "status":"ERROR", "error": repr(e)})

df_diag = pd.DataFrame(rows)
display(df_diag)


,mes,lic_prov,oc_prov,lic_org,oc_org,prov_lic,prov_oc,prov_inter,prov_pct_lic,org_lic,org_oc,org_inter,org_pct_lic
0,2024-10,CodigoProveedor,CodigoProveedor,CodigoOrganismo,CodigoOrganismoPublico,13672,22209,6722,49.166179,888,1038,881,99.211712
1,2024-11,CodigoProveedor,CodigoProveedor,CodigoOrganismo,CodigoOrganismoPublico,8012,22784,4670,58.287569,775,1035,769,99.225806
2,2024-12,CodigoProveedor,CodigoProveedor,CodigoOrganismo,CodigoOrganismoPublico,9418,22470,5029,53.397749,826,1041,821,99.394673


## 8 — Export de evidencia

Se guardan tablas en:
`BASE_DIR/04_Hallazgos/diagnostico_cruce_lic_oc_3m/`


In [8]:
# 8 - EXPORT (evidencias)
OUT_DIR = os.path.join(BASE_DIR, "04_Hallazgos", "diagnostico_cruce_lic_oc_3m")
os.makedirs(OUT_DIR, exist_ok=True)

df_exact.to_csv(os.path.join(OUT_DIR, "match_exact_codigo_lic_oc.csv"), index=False)
df_text.to_csv(os.path.join(OUT_DIR, "match_text_extraccion_oc.csv"), index=False)
df_diag.to_csv(os.path.join(OUT_DIR, "diag_proveedor_organismo.csv"), index=False)

print("OK - exportado en:", OUT_DIR)
print(" - match_exact_codigo_lic_oc.csv")
print(" - match_text_codigo_lic_oc.csv")
print(" - diag_proveedor_organismo.csv")


OK - exportado en: /home/engineer/Documents/Proyectos/TFM_CSV_Biblioteca_Readme/04_Hallazgos/diagnostico_cruce_lic_oc_3m
 - match_exact_codigo_lic_oc.csv
 - match_text_codigo_lic_oc.csv
 - diag_proveedor_organismo.csv
